In [1]:
from typing import Sequence

import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
from jaxtyping import Float, Int

# custom utils
from muutils.json_serialize import (
	SerializableDataclass,
	serializable_dataclass,
	serializable_field,
)
from zanj.torchutil import ConfiguredModel, set_config_class
from trnbl import TrainingManager
from trnbl.loggers.local import LocalLogger


from attention_motifs.ae import AttnAEConfig, AttnAE
from attention_motifs.dataset.dataset import CollectedAttentionPatternDataloader

f:\projects\attention-motifs\.venv\Lib\site-packages\trnbl\loggers\base.py:17: UserWarning: GPUtil not available: No module named 'GPUtil'
  warnings.warn(f"GPUtil not available: {e}")


In [2]:
# magic autoreload
%load_ext autoreload
%autoreload 2

In [3]:
config: AttnAEConfig = AttnAEConfig(
	min_size=64,
	max_size=512,
	latent_dim=64,
)

In [4]:
model: AttnAE = AttnAE(config)

TypeError: _MaxPoolNd.__init__() missing 1 required positional argument: 'kernel_size'

In [ ]:





train_loader: torch.utils.data.DataLoader
val_loader: torch.utils.data.DataLoader | None = None
num_epochs: int = 100
learning_rate: float = 1e-3
recon_weight: float = 1.0
contrast_weight: float = 1.0
project_name: str = "contrastive-ae"
checkpoint_interval: str = "1/10 run"
eval_interval: str = "1k samples"
device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = model.to(device)
optimizer: torch.optim.Optimizer = model.config.optimizer(
	model.parameters(),
	lr=learning_rate,
)

# setup logger
logger: LocalLogger = LocalLogger(
	project=project_name,
	metric_names=[
		"train/loss",
		"train/recon_loss",
		"train/contrast_loss",
		"val/loss",
		"val/recon_loss",
		"val/contrast_loss",
	],
	train_config=dict(
		model_config=model.zanj_model_config.serialize(),
		learning_rate=learning_rate,
		recon_weight=recon_weight,
		contrast_weight=contrast_weight,
	),
)


In [ ]:

def evaluation_step(model: AttnAE) -> dict[str, float]:
	"""Evaluate model on validation set"""
	if val_loader is None:
		return {}

	model.eval()
	val_metrics = {"val/loss": 0.0, "val/recon_loss": 0.0, "val/contrast_loss": 0.0}

	with torch.no_grad():
		for batch_idx, (x, index_tuple) in enumerate(val_loader):
			x = x.to(device)
			index_tuple = tuple(i.to(device) for i in index_tuple)

			x_recon, z = model(x)
			recon_loss = F.mse_loss(x_recon, x)

			batch_size = x.size(0)
			z1 = z.repeat_interleave(batch_size, dim=0)
			z2 = z.repeat(batch_size, 1)
			idx1 = tuple(i.repeat_interleave(batch_size) for i in index_tuple)
			idx2 = tuple(i.repeat(batch_size) for i in index_tuple)
			contrast_loss = model.contrastive_loss(z1, z2, idx1, idx2)

			total_loss = recon_weight * recon_loss + contrast_weight * contrast_loss

			val_metrics["val/loss"] += total_loss.item()
			val_metrics["val/recon_loss"] += recon_loss.item()
			val_metrics["val/contrast_loss"] += contrast_loss.item()

	for k in val_metrics:
		val_metrics[k] /= len(val_loader)

	model.train()
	return val_metrics


In [ ]:

with TrainingManager(
	model=model,
	logger=logger,
	evals={
		eval_interval: evaluation_step,
	}.items(),
	checkpoint_interval=checkpoint_interval,
) as tr:
	for epoch in tr.epoch_loop(range(num_epochs)):
		for x, index_tuple in tr.batch_loop(train_loader):
			x = x.to(device)
			index_tuple = tuple(i.to(device) for i in index_tuple)

			optimizer.zero_grad()
			x_recon, z = model(x)

			# reconstruction loss
			recon_loss = F.mse_loss(x_recon, x)

			# contrastive loss using all pairs in batch
			batch_size = x.size(0)
			z1 = z.repeat_interleave(batch_size, dim=0)
			z2 = z.repeat(batch_size, 1)
			idx1 = tuple(i.repeat_interleave(batch_size) for i in index_tuple)
			idx2 = tuple(i.repeat(batch_size) for i in index_tuple)
			contrast_loss = model.contrastive_loss(z1, z2, idx1, idx2)

			# combined loss and backward pass
			total_loss = recon_weight * recon_loss + contrast_weight * contrast_loss
			total_loss.backward()
			optimizer.step()

			# log metrics
			tr.batch_update(
				samples=len(x),
				**{
					"train/loss": total_loss.item(),
					"train/recon_loss": recon_loss.item(),
					"train/contrast_loss": contrast_loss.item(),
				},
			)

